# Session 3 — Track A: Production RAG Service (Instructor Capstone)

In this capstone, students build a **production-grade RAG service** from scratch, integrating everything from Days 1-3: embeddings, vector stores, chunking, query transformation, caching, monitoring, and evaluation.

> **Instructor Note:** This notebook contains fully solved versions of all 4 milestones with detailed approach explanations. Use this as a reference during the lab and for students who get stuck.

## Capstone Objectives

By the end of this capstone, students will have built:

1. A **document ingestion pipeline** with smart chunking and metadata
2. A **retrieval engine** with query expansion and reranking
3. A **generation layer** with citations, streaming, and guardrails
4. A **production wrapper** with caching, monitoring, and evaluation

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

import openai
import chromadb
import json
import time
import hashlib
import numpy as np
from datetime import datetime
from collections import defaultdict
from typing import List, Dict, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter

print("All imports successful!")

---
## Milestone 1: Document Ingestion Pipeline

Build a pipeline that takes raw documents, applies smart chunking, embeds, and indexes them in ChromaDB.

> **Approach:** We create a class that wraps the full ingestion flow. Documents are chunked with `RecursiveCharacterTextSplitter`, and each chunk inherits metadata from its parent doc (source, topic) plus gets new metadata (chunk_index, total_chunks). All chunks are batch-embedded and stored in ChromaDB.

In [ ]:
# Milestone 1 — SOLUTION: Document Ingestion Pipeline

class DocumentIngester:
    def __init__(self, collection_name="rag_capstone", chunk_size=300, chunk_overlap=50):
        self.embed_client = openai.OpenAI()
        self.chroma = chromadb.Client()
        self.collection = self.chroma.create_collection(
            name=collection_name, metadata={"hnsw:space": "cosine"}
        )
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap,
            separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " "]
        )

    def ingest(self, documents: List[Dict]) -> Dict:
        """Ingest documents. Each doc: {text, source, metadata}."""
        all_chunks = []
        all_metadatas = []
        all_ids = []
        
        for doc_idx, doc in enumerate(documents):
            chunks = self.splitter.split_text(doc["text"])
            for chunk_idx, chunk_text in enumerate(chunks):
                chunk_id = f"doc{doc_idx}_chunk{chunk_idx}"
                meta = {
                    "source": doc.get("source", "unknown"),
                    "chunk_index": chunk_idx,
                    "total_chunks": len(chunks),
                    **doc.get("metadata", {})
                }
                all_chunks.append(chunk_text)
                all_metadatas.append(meta)
                all_ids.append(chunk_id)
        
        # Batch embed
        embeddings = [
            e.embedding for e in self.embed_client.embeddings.create(
                model="text-embedding-3-small", input=all_chunks
            ).data
        ]
        
        # Store in ChromaDB
        self.collection.add(
            documents=all_chunks, embeddings=embeddings,
            ids=all_ids, metadatas=all_metadatas
        )
        
        stats = {
            "documents_ingested": len(documents),
            "total_chunks": len(all_chunks),
            "avg_chunk_size": round(np.mean([len(c) for c in all_chunks]), 1),
            "collection_size": self.collection.count()
        }
        print(f"Ingested {stats['documents_ingested']} docs -> {stats['total_chunks']} chunks")
        return stats


# Test
test_docs = [
    {"text": "RAG combines retrieval with generation to ground LLM answers in external knowledge. It was introduced by Lewis et al. in 2020. The key insight is that LLMs can produce more accurate and verifiable answers when given relevant context from a knowledge base.", "source": "rag_guide.md", "metadata": {"topic": "rag"}},
    {"text": "ChromaDB is an open-source embedding database designed for AI applications. It supports cosine similarity and Euclidean distance metrics. ChromaDB can run in-memory for development or persist to disk for production use.", "source": "vectordb.md", "metadata": {"topic": "infrastructure"}},
    {"text": "Chunking strategy directly affects retrieval quality. Small chunks (100-200 tokens) provide precision but lose context. Large chunks (500-1000 tokens) preserve context but dilute relevance. The sweet spot is typically 200-500 tokens with 10-20% overlap.", "source": "best_practices.md", "metadata": {"topic": "rag"}},
    {"text": "Query transformation improves retrieval for complex or ambiguous questions. Multi-query expansion generates alternative formulations. HyDE generates a hypothetical answer and uses it as the search query. Both techniques increase recall at the cost of latency.", "source": "advanced_rag.md", "metadata": {"topic": "rag"}},
    {"text": "Reranking rescores retrieved results using a more powerful model. Cross-encoders like BGE-reranker process the query-document pair together, producing more accurate relevance scores than bi-encoder similarity. This improves precision significantly.", "source": "advanced_rag.md", "metadata": {"topic": "rag"}}
]

ingester = DocumentIngester()
stats = ingester.ingest(test_docs)
print(stats)

---
## Milestone 2: Advanced Retrieval Engine

Build a retrieval engine with query expansion, deduplication, and LLM reranking.

> **Approach:** We expand each query into 3 variants using the LLM, retrieve candidates for each from ChromaDB, deduplicate by document text, then use the LLM to score relevance 0-10. The final results are sorted by score.

In [ ]:
# Milestone 2 — SOLUTION: Advanced Retrieval Engine

class RetrievalEngine:
    def __init__(self, collection):
        self.collection = collection
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.embed_client = openai.OpenAI()

    def _expand_query(self, query, n=3):
        response = self.llm.invoke([
            SystemMessage(content=f"Generate {n} alternative versions of this question for search. Return as a JSON list of strings."),
            HumanMessage(content=query)
        ])
        try:
            alts = json.loads(response.content)
        except:
            alts = []
        return [query] + alts

    def _retrieve_raw(self, queries, per_query_k=4):
        seen = set()
        candidates = []
        for q in queries:
            q_emb = self.embed_client.embeddings.create(
                model="text-embedding-3-small", input=[q]
            ).data[0].embedding
            results = self.collection.query(query_embeddings=[q_emb], n_results=per_query_k)
            for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
                if doc not in seen:
                    seen.add(doc)
                    candidates.append({"text": doc, "metadata": meta})
        return candidates

    def _rerank(self, query, candidates):
        scored = []
        for c in candidates:
            response = self.llm.invoke([
                SystemMessage(content="Rate relevance 0-10. Return ONLY the number."),
                HumanMessage(content=f"Question: {query}\nDocument: {c['text']}")
            ])
            try:
                score = int(response.content.strip())
            except:
                score = 5
            scored.append({**c, "score": score})
        scored.sort(key=lambda x: x["score"], reverse=True)
        return scored

    def retrieve(self, query, k=5) -> List[Dict]:
        queries = self._expand_query(query)
        candidates = self._retrieve_raw(queries)
        ranked = self._rerank(query, candidates)
        return ranked[:k]


# Test
engine = RetrievalEngine(ingester.collection)
results = engine.retrieve("How does RAG work?")
print("Retrieved chunks:")
for r in results:
    print(f"  {r['score']}/10 [{r['metadata']['source']}] {r['text'][:70]}...")

---
## Milestone 3: Generation Layer with Citations

Build the generation component that produces cited answers with confidence flagging.

> **Approach:** We format retrieved chunks with source tags, instruct the LLM to cite sources inline, and detect low-confidence answers by checking for hedging phrases. The system prompt explicitly requires citation of sources.

In [ ]:
# Milestone 3 — SOLUTION: Generation Layer with Citations

class RAGGenerator:
    def __init__(self, model="gpt-4o-mini"):
        self.llm = ChatOpenAI(model=model, temperature=0)
        self.system_prompt = """Answer the question based ONLY on the provided context.
Rules:
1. Cite sources using [Source: filename] format
2. If the context doesn't contain enough information, say "I don't have sufficient information to answer this fully."
3. Do not make up information beyond what the context provides
4. Be concise and direct"""

    def generate(self, question, retrieved_chunks) -> Dict:
        # Format context with source tags
        context_parts = []
        sources_used = set()
        for chunk in retrieved_chunks:
            source = chunk['metadata'].get('source', 'unknown')
            sources_used.add(source)
            context_parts.append(f"[Source: {source}] {chunk['text']}")
        context = "\n\n".join(context_parts)
        
        # Generate
        start = time.time()
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}")
        ])
        latency = (time.time() - start) * 1000
        
        content = response.content
        
        # Confidence detection
        low_confidence_phrases = ["i don't have", "insufficient information", "not enough context", "cannot determine"]
        confidence = "low" if any(p in content.lower() for p in low_confidence_phrases) else "high"
        
        return {
            "content": content,
            "sources": list(sources_used),
            "confidence": confidence,
            "latency_ms": round(latency, 2),
            "context_chunks": len(retrieved_chunks)
        }


# Test
generator = RAGGenerator()
answer = generator.generate("How does RAG work and what are its benefits?", results)
print(f"Answer: {answer['content']}")
print(f"\nSources: {answer['sources']}")
print(f"Confidence: {answer['confidence']}")
print(f"Latency: {answer['latency_ms']}ms")

---
## Milestone 4: Production Wrapper — Caching, Monitoring & Evaluation

Wrap everything in a production service with caching, logging, cost tracking, and quality evaluation.

> **Approach:** We compose all three milestone components into a single service class. The pipeline is: check cache -> retrieve -> generate -> evaluate -> cache & log. The dashboard aggregates cache hit rate, average latency, average quality score, and cost.

In [ ]:
# Milestone 4 — SOLUTION: Production RAG Service

class ProductionRAGService:
    def __init__(self, documents):
        # Initialize components
        self.ingester = DocumentIngester(collection_name="prod_rag")
        self.ingester.ingest(documents)
        self.retriever = RetrievalEngine(self.ingester.collection)
        self.generator = RAGGenerator()
        self.eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        
        # Cache and monitoring
        self.cache = {}  # hash -> response
        self.logs = []
        self.quality_scores = []
    
    def _hash(self, text):
        return hashlib.md5(text.strip().lower().encode()).hexdigest()
    
    def _evaluate(self, question, answer, context_chunks):
        context = " ".join([c['text'] for c in context_chunks])
        metrics = {}
        for metric, prompt in {
            "relevance": f"Rate 1-5: How relevant is the context to the question?\nQuestion: {question}\nContext: {context[:500]}",
            "faithfulness": f"Rate 1-5: Is the answer supported by the context?\nContext: {context[:500]}\nAnswer: {answer}",
            "completeness": f"Rate 1-5: Does the answer fully address the question?\nQuestion: {question}\nAnswer: {answer}"
        }.items():
            resp = self.eval_llm.invoke([
                SystemMessage(content="Return ONLY a number 1-5."),
                HumanMessage(content=prompt)
            ])
            try:
                metrics[metric] = int(resp.content.strip())
            except:
                metrics[metric] = 3
        metrics["overall"] = round(np.mean(list(metrics.values())), 2)
        return metrics
    
    def query(self, question) -> Dict:
        start = time.time()
        
        # Check cache
        key = self._hash(question)
        if key in self.cache:
            cached = self.cache[key]
            latency = (time.time() - start) * 1000
            self.logs.append({"question": question, "cached": True, "latency_ms": latency})
            return {**cached, "cached": True, "latency_ms": round(latency, 2)}
        
        # Retrieve
        chunks = self.retriever.retrieve(question, k=3)
        
        # Generate
        answer = self.generator.generate(question, chunks)
        
        # Evaluate
        metrics = self._evaluate(question, answer["content"], chunks)
        self.quality_scores.append(metrics["overall"])
        
        latency = (time.time() - start) * 1000
        result = {
            "answer": answer["content"],
            "sources": answer["sources"],
            "confidence": answer["confidence"],
            "metrics": metrics,
            "cached": False,
            "latency_ms": round(latency, 2)
        }
        
        # Cache and log
        self.cache[key] = result
        self.logs.append({"question": question, "cached": False,
                          "latency_ms": latency, "quality": metrics["overall"]})
        
        return result
    
    def dashboard(self) -> Dict:
        total = len(self.logs)
        cached = sum(1 for l in self.logs if l.get("cached"))
        return {
            "total_queries": total,
            "cache_hit_rate": f"{cached/total*100:.1f}%" if total else "N/A",
            "avg_latency_ms": round(np.mean([l["latency_ms"] for l in self.logs]), 2) if self.logs else 0,
            "avg_quality": round(np.mean(self.quality_scores), 2) if self.quality_scores else 0,
            "cache_size": len(self.cache)
        }


# Test
service = ProductionRAGService(test_docs)

questions = [
    "How does RAG work?",
    "What is the recommended chunk size?",
    "How does RAG work?",  # cached
    "What is reranking?"
]

for q in questions:
    result = service.query(q)
    cached_str = "CACHED" if result["cached"] else f"quality={result['metrics']['overall']}"
    print(f"[{cached_str:12s}] {q[:40]:40s} | {result['latency_ms']:.0f}ms")
    print(f"  Answer: {result['answer'][:100]}...\n")

print("--- Dashboard ---")
for k, v in service.dashboard().items():
    print(f"  {k}: {v}")

---
## Capstone Summary

Students built a **production-grade RAG service** with:

1. **Ingestion** -- Smart chunking with metadata preservation
2. **Retrieval** -- Multi-query expansion and LLM reranking
3. **Generation** -- Cited answers with confidence flagging
4. **Production** -- Caching, monitoring, and automated evaluation

**Up Next:** Session 4 — Cross-track presentations, governance, and closing.